In [1]:
from __future__ import annotations

import math
import random
from dataclasses import dataclass
from typing import Optional, Literal

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim


# ============================================================
# Utilities
# ============================================================

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def to_onehot(x: np.ndarray, n: int) -> np.ndarray:
    x_onehot = np.zeros((len(x), n), dtype=np.float32)
    x_onehot[np.arange(len(x)), x] = 1.0
    return x_onehot


# ============================================================
# 2ACDC task sequences
# ============================================================

# 0: teleportation
# 1: grey region
# 2: near cue
# 3: far cue
# 4: near reward cue
# 5: far reward cue
# 6: reward

TRIAL_NEAR = np.array([1,1,1,1,1,1,2,2,2,2,1,1,1,4,6,1,1,1,5,5,1,1,0], dtype=np.int64)
TRIAL_FAR  = np.array([1,1,1,1,1,1,3,3,3,3,1,1,1,4,4,1,1,1,5,6,1,1,0], dtype=np.int64)

TRIAL_LEN = len(TRIAL_NEAR)
OBS = len(np.unique(np.concatenate((TRIAL_NEAR, TRIAL_FAR))))  # should be 7


def sample_trials(
    num_trials: int,
    ensure_both_types: bool = True,
) -> np.ndarray:
    """
    Returns an array of shape (num_trials,) with entries 0=near, 1=far.
    """
    while True:
        trials = np.random.choice(2, num_trials)
        if not ensure_both_types:
            return trials
        if np.sum(trials == 0) > 1 and np.sum(trials == 1) > 1:
            return trials


def build_trial_stream(trials: np.ndarray) -> np.ndarray:
    """
    Concatenate trials into one long sequence of token ids.
    """
    x = np.zeros(len(trials) * TRIAL_LEN, dtype=np.int64)
    for i, tr in enumerate(trials):
        if tr == 0:
            x[i * TRIAL_LEN:(i + 1) * TRIAL_LEN] = TRIAL_NEAR
        else:
            x[i * TRIAL_LEN:(i + 1) * TRIAL_LEN] = TRIAL_FAR
    return x


# ============================================================
# Custom RNN cell
# ============================================================

ActivationType = Literal["exp_softmax", "poly_softmax", "relu", "sigmoid"]


class SoftmaxStyleRNNCell(nn.Module):
    """
    A custom vanilla RNN cell closely matching your uploaded code.

    hidden_t = normalize( phi( W_ih x_t + W_hh h_{t-1} ) )

    For:
      - exp_softmax: phi(z) = exp(z / temperature)
      - poly_softmax: phi(z) = relu(z)^power
      - relu: phi(z) = relu(z)
      - sigmoid: phi(z) = sigmoid(z)

    Normalization is only applied for the two 'softmax-style' modes by default.
    """

    def __init__(
        self,
        input_size: int,
        hidden_size: int,
        activation: ActivationType = "exp_softmax",
        temperature: float = 1.0,
        poly_power: float = 8.0,
        normalize_relu_sigmoid: bool = False,
    ):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.activation = activation
        self.temperature = temperature
        self.poly_power = poly_power
        self.normalize_relu_sigmoid = normalize_relu_sigmoid

        self.W_ih = nn.Parameter(torch.empty(hidden_size, input_size))
        self.W_hh = nn.Parameter(torch.empty(hidden_size, hidden_size))

    def reset_parameters(
        self,
        ih_std: float = 0.001,
        hh_std: float = 0.001,
    ) -> None:
        with torch.no_grad():
            self.W_ih.copy_(torch.tensor(
                np.random.normal(0.0, ih_std, (self.hidden_size, self.input_size)),
                dtype=torch.float32
            ))
            self.W_hh.copy_(torch.tensor(
                np.random.normal(0.0, hh_std, (self.hidden_size, self.hidden_size)),
                dtype=torch.float32
            ))

    def forward(self, x: torch.Tensor, hidden: torch.Tensor) -> torch.Tensor:
        # x: (input_size,)
        # hidden: (hidden_size,)
        ff = torch.mm(x.unsqueeze(0), self.W_ih.t())
        rec = torch.mm(hidden.unsqueeze(0), self.W_hh.t())
        logits = (ff + rec).squeeze(0)

        if self.activation == "exp_softmax":
            new_hidden = torch.exp(logits / self.temperature)
            new_hidden = new_hidden / (torch.sum(new_hidden) + 1e-12)

        elif self.activation == "poly_softmax":
            new_hidden = torch.relu(logits).pow(self.poly_power)
            denom = torch.sum(new_hidden)
            if denom.item() == 0:
                new_hidden = torch.full_like(new_hidden, 1.0 / len(new_hidden))
            else:
                new_hidden = new_hidden / (denom + 1e-12)

        elif self.activation == "relu":
            new_hidden = torch.relu(logits)
            if self.normalize_relu_sigmoid:
                new_hidden = new_hidden / (torch.sum(new_hidden) + 1e-12)

        elif self.activation == "sigmoid":
            new_hidden = torch.sigmoid(logits)
            if self.normalize_relu_sigmoid:
                new_hidden = new_hidden / (torch.sum(new_hidden) + 1e-12)

        else:
            raise ValueError(f"Unknown activation: {self.activation}")

        return new_hidden


# ============================================================
# Full RNN model
# ============================================================

class SoftmaxStyleRNN(nn.Module):
    """
    Forward signature mirrors your reference:
      input:  x of shape (T, input_size)
      output: logits of shape (T, output_size), hidden states of shape (T, hidden_size)
    """

    def __init__(
        self,
        input_size: int,
        hidden_size: int,
        output_size: int,
        activation: ActivationType = "exp_softmax",
        temperature: float = 1.0,
        poly_power: float = 8.0,
        normalize_relu_sigmoid: bool = False,
        ih_std: float = 0.001,
        hh_std: float = 0.001,
        ho_std: float = 1.0,
    ):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size

        self.rnn_cell = SoftmaxStyleRNNCell(
            input_size=input_size,
            hidden_size=hidden_size,
            activation=activation,
            temperature=temperature,
            poly_power=poly_power,
            normalize_relu_sigmoid=normalize_relu_sigmoid,
        )
        self.rnn_cell.reset_parameters(ih_std=ih_std, hh_std=hh_std)

        self.fc = nn.Linear(hidden_size, output_size, bias=False)
        with torch.no_grad():
            self.fc.weight.copy_(torch.tensor(
                np.random.normal(0.0, ho_std, (output_size, hidden_size)),
                dtype=torch.float32
            ))

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        """
        x: (T, input_size)
        returns:
          out: (T, output_size)
          hidden_all: (T, hidden_size)
        """
        T = x.size(0)
        hidden_all = torch.zeros(T, self.hidden_size, dtype=x.dtype, device=x.device)

        h0 = torch.zeros(self.hidden_size, dtype=x.dtype, device=x.device)
        hidden_all[0, :] = self.rnn_cell(x[0, :], h0)

        for t in range(1, T):
            hidden_all[t, :] = self.rnn_cell(x[t, :], hidden_all[t - 1, :].clone())

        out = self.fc(hidden_all)
        return out, hidden_all


# ============================================================
# Correlation analysis, kept close to your reference
# ============================================================

def corr_finder(hidden_all: torch.Tensor, test_trials: np.ndarray, tr_len: int, hidden_size: int) -> np.ndarray:
    """
    Replicates the logic of the uploaded analysis code.
    """
    hidden_all = hidden_all.detach().cpu().numpy()

    test0 = np.where(test_trials == 0)[0]
    test0_act = np.zeros((tr_len, hidden_size, len(test0)), dtype=np.float32)
    for count, i in enumerate(test0):
        test0_act[:, :, count] = hidden_all[i * tr_len:(i + 1) * tr_len, :]

    test1 = np.where(test_trials == 1)[0]
    test1_act = np.zeros((tr_len, hidden_size, len(test1)), dtype=np.float32)
    for count, i in enumerate(test1):
        test1_act[:, :, count] = hidden_all[i * tr_len:(i + 1) * tr_len, :]

    corrplot = np.corrcoef(np.mean(test0_act, axis=2), np.mean(test1_act, axis=2))
    return corrplot


# ============================================================
# Training config
# ============================================================

@dataclass
class SimulationConfig:
    num_trials: int = 100
    num_trials_train: int = 50
    hidden_size: int = 5000
    epochs: int = 500
    save_each: int = 10
    lr: float = 0.2

    activation: ActivationType = "exp_softmax"
    temperature: float = 1.0
    poly_power: float = 8.0
    normalize_relu_sigmoid: bool = False

    ih_std: float = 0.001
    hh_std: float = 0.001
    ho_std: float = 1.0

    device: str = "cuda" if torch.cuda.is_available() else "cpu"


# ============================================================
# Single simulation
# ============================================================

def run_single_simulation(seed: int, cfg: SimulationConfig) -> dict[str, np.ndarray]:
    set_seed(seed)
    device = torch.device(cfg.device)

    model = SoftmaxStyleRNN(
        input_size=OBS,
        hidden_size=cfg.hidden_size,
        output_size=OBS,
        activation=cfg.activation,
        temperature=cfg.temperature,
        poly_power=cfg.poly_power,
        normalize_relu_sigmoid=cfg.normalize_relu_sigmoid,
        ih_std=cfg.ih_std,
        hh_std=cfg.hh_std,
        ho_std=cfg.ho_std,
    ).to(device)

    optimizer = optim.Adam(model.parameters(), lr=cfg.lr)
    loss_func = nn.CrossEntropyLoss()

    n_save = cfg.epochs // cfg.save_each
    loss_all = np.zeros(n_save, dtype=np.float32)
    accuracy_curve_all_test = np.zeros(n_save, dtype=np.float32)
    corr_curve = np.zeros((n_save, TRIAL_LEN * 2, TRIAL_LEN * 2), dtype=np.float32)

    save_idx = 0

    for epoch in range(cfg.epochs):
        trials = sample_trials(cfg.num_trials, ensure_both_types=True)
        x_tokens = build_trial_stream(trials)
        x_onehot = to_onehot(x_tokens, OBS)

        train_x = torch.tensor(
            x_onehot[:cfg.num_trials_train * TRIAL_LEN],
            dtype=torch.float32,
            device=device,
        )
        test_x = torch.tensor(
            x_onehot[cfg.num_trials_train * TRIAL_LEN:],
            dtype=torch.float32,
            device=device,
        )

        # next-step prediction on the training stream
        optimizer.zero_grad()
        input_x = train_x[:-1]
        prediction, hidden_all = model(input_x)
        target = train_x[1:].argmax(dim=1)
        loss = loss_func(prediction, target)
        loss.backward()
        optimizer.step()

        if epoch % cfg.save_each == 0:
            with torch.no_grad():
                pred_test, hidden_test = model(test_x)

                actual_reward = np.where(test_x[1:].cpu().argmax(dim=1).numpy() == 6)[0]
                predicted_reward = np.where(pred_test.cpu().argmax(dim=1).numpy() == 6)[0]
                accuracy = len(np.intersect1d(actual_reward, predicted_reward)) / max(len(actual_reward), 1)

                accuracy_curve_all_test[save_idx] = accuracy
                corr_curve[save_idx] = corr_finder(
                    hidden_test, trials[cfg.num_trials_train:], TRIAL_LEN, cfg.hidden_size
                )
                loss_all[save_idx] = float(loss.item())
                save_idx += 1

    return {
        "loss_all": loss_all,
        "accuracy_curve_all_test": accuracy_curve_all_test,
        "corr_curve": corr_curve,
    }


# ============================================================
# Multi-seed wrapper
# ============================================================

def run_many_simulations(
    seeds: list[int],
    cfg: SimulationConfig,
) -> dict[str, np.ndarray]:
    all_loss = []
    all_acc = []
    all_corr = []

    for i, seed in enumerate(seeds):
        print(f"Running seed {seed} ({i + 1}/{len(seeds)})")
        res = run_single_simulation(seed, cfg)
        all_loss.append(res["loss_all"])
        all_acc.append(res["accuracy_curve_all_test"])
        all_corr.append(res["corr_curve"])

    combined_loss_all = np.stack(all_loss, axis=0)
    combined_accuracy_curve = np.stack(all_acc, axis=0)
    combined_corr_curve = np.stack(all_corr, axis=0)

    print("\nFinal Statistics")
    print(f"Average loss: {np.mean(combined_loss_all[:, -1]):.4f}")
    print(f"Average accuracy: {np.mean(combined_accuracy_curve[:, -1]):.4f}")

    return {
        "loss_all": combined_loss_all,
        "accuracy_curve_all_test": combined_accuracy_curve,
        "corr_curve": combined_corr_curve,
    }


# ============================================================
# Example usage
# ============================================================

if __name__ == "__main__":
    cfg = SimulationConfig(
        num_trials=100,
        num_trials_train=50,
        hidden_size=5000,
        epochs=500,
        save_each=10,
        lr=0.2,
        activation="exp_softmax",   # or "poly_softmax", "relu", "sigmoid"
        temperature=1.0,
        poly_power=8.0,
        normalize_relu_sigmoid=False,
        ih_std=0.001,
        hh_std=0.001,
        ho_std=1.0,
        device="cuda" if torch.cuda.is_available() else "cpu",
    )

    seeds = list(range(200, 204))
    results = run_many_simulations(seeds, cfg)

    print("loss_all shape:", results["loss_all"].shape)
    print("accuracy_curve_all_test shape:", results["accuracy_curve_all_test"].shape)
    print("corr_curve shape:", results["corr_curve"].shape)

Running seed 200 (1/4)
Running seed 201 (2/4)
Running seed 202 (3/4)
Running seed 203 (4/4)

Final Statistics
Average loss: 0.0430
Average accuracy: 0.8700
loss_all shape: (4, 50)
accuracy_curve_all_test shape: (4, 50)
corr_curve shape: (4, 50, 46, 46)
